# step06 - Hydrographic sections (cross-shelf)

Vertical sections along a transect, built from the **1 m downcast** products in `L2/CTD/CTD_1m_down/`
(and the merged SUNA product for nitrate). A variable is contoured with **along-track distance** on the
x axis and **pressure** on the y axis, with the seabed shaded and the stations marked.

**Positions** come from the cruise cast-summary (`<CRUISE_ID>_CTD_cast_summary.csv`): columns
`cast, date_time_utc_start, latitude_start, longitude_start, station, ...`. step06 uses the
**start-of-cast fix** only (the `_end` fix is set aside for now; a dropped-to-zero latitude is treated
as missing), and maps the integer `cast` to the matching `*_1m_down.cnv`. A plain `stations.csv`
(`cast_id, lat_deg, lon_deg`) is also accepted.

**Grouping into transects (`GROUP_BY`)**
- `"date"` (default): one section per **day**, from `date_time_utc_start` - each day's casts are a
  transect, ordered cross-shelf by position. This matches how the lines were run.
- `"line"`: one section per **station line**, grouping by the non-numeric key of the station label
  (`S1H..S12H` -> `SH`, `P1..P3` -> `P`).
- `"all"`: force every cast into one section.

A group with fewer than `MIN_GROUP_CASTS` casts, or spanning less than `MIN_SPAN_KM` along-track (a
single spot / repeat station), is skipped. Restrict with `DATES_TO_SECTION` or `LINES_TO_SECTION`.

**How the grid is made**
1. Fit the group's axis to its station positions (principal axis), project each station onto it for
   along-track distance, orient so distance grows **offshore** (deeper). Per-station cross-track offset
   is reported, so an off-line station is visible.
2. Grid each variable by **linear interpolation on a triangulation** of all (distance, pressure) data
   points (matplotlib only, no SciPy), then mask below the seabed.

**Honest notes**
- The colour between stations is **interpolation, a visual aid**, not measured data. Station ticks mark
  the real casts.
- Pressure (dbar) is the vertical coordinate (within ~1 % of depth in metres on the shelf).
- Oxygen is **provisional** (Tau off). Nitrate appears only where a cast has a merged SUNA product.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation, LinearTriInterpolator

# ------------------------------------------------------------------ config
CRUISE_ID = "P45_07"                       # <-- the only switch to change per cruise
CTD_ROOT  = Path(r"C:\Projects\ctd_pipeline")

CRUISE_DIR    = CTD_ROOT / "cruises" / CRUISE_ID
ONE_DOWN_DIR  = CRUISE_DIR / "L2" / "CTD" / "CTD_1m_down"
SUNA_DIR      = CRUISE_DIR / "L2" / "SUNA"
META_DIR      = CRUISE_DIR / "metadata"
STATION_CANDIDATES = [f"{CRUISE_ID}_CTD_cast_summary.csv", "stations.csv", f"{CRUISE_ID}_stations.csv"]
OUT_DIR       = CRUISE_DIR / "_audit" / "sections"; OUT_DIR.mkdir(parents=True, exist_ok=True)

VARS = ["temperature", "salinity", "sigma_theta", "oxygen", "nitrate"]

# how casts are grouped into one section each
GROUP_BY         = "date"     # "date" (one per day) | "line" (per station line) | "all"
DATES_TO_SECTION = []         # e.g. ["2026-07-29","2026-08-04"]; [] = every day
LINES_TO_SECTION = []         # e.g. ["SH"]; [] = every detected line  (used when GROUP_BY="line")
SECTION_ALL      = False      # True = ignore grouping, one section of all casts
MANUAL_CAST_ORDER = []        # explicit cast_id order for a single section (overrides grouping)
MIN_GROUP_CASTS  = 3          # a group needs at least this many casts
MIN_SPAN_KM      = 3.0        # skip a group spanning less than this (a single spot / repeat station)
CROSS_WARN_KM    = 5.0        # warn if a station sits more than this far off the fitted line
GRID_DZ = 1.0; GRID_NX = 240

VAR_STYLE = {
    "temperature": dict(cmap="RdYlBu_r", unit="deg C",   label="Temperature"),
    "salinity":    dict(cmap="viridis",  unit="PSU",     label="Salinity"),
    "sigma_theta": dict(cmap="cividis",  unit="kg/m3",   label="Sigma-theta"),
    "oxygen":      dict(cmap="YlGnBu",   unit="",         label="Oxygen (provisional)"),
    "nitrate":     dict(cmap="YlGnBu",   unit="umol/L",  label="Nitrate (L3)"),
}
print("cruise:", CRUISE_ID, "| group by:", GROUP_BY, "| out:", OUT_DIR)

In [ ]:
# ------------------------------------------------------------------ readers (1 m profiles)
BADLO, BADHI = -1.2e-28, -8e-29

def _parse_cnv(path):
    txt = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, ln in enumerate(txt):
        s = ln.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(ln)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    if ds is None: return None
    cols = [names[k] for k in sorted(names)]
    rows = [l.split()[:len(cols)] for l in txt[ds:] if l.strip() and len(l.split()) >= len(cols)]
    df = pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
    bad = ((df.le(BADHI) & df.ge(BADLO)) | (df.abs() > 1e30))
    if "flag" in bad.columns: bad["flag"] = False
    return df.mask(bad)

def _pick(df, *cands):
    if df is None: return None
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None

_C = dict(
    P=("prDM","prdM","prSM","pr"),
    temperature=("tv290C","t090C","t190C","tv290","temperature"),
    salinity=("sal00","sal78","sal11","salinity"),
    sigma_theta=("sigma-\u00e9","sigma-t","sigma-\u03b8","sigma","density"),
    oxygen=("sbeox0Mm/Kg","sbeox0ML/L","sbeox0Mm/L","sbox0Mm/Kg","oxygen","sbeox"),
)

def cast_ids():
    if not ONE_DOWN_DIR.exists(): return []
    return sorted({re.sub(r"_1m.*$", "", p.stem) for p in ONE_DOWN_DIR.glob("*.cnv")})

def _file_for(cast):
    hits = sorted(ONE_DOWN_DIR.glob(f"{cast}*.cnv"))
    return hits[0] if hits else None

def load_profile(cast):
    f = _file_for(cast)
    if f is None: return None
    df = _parse_cnv(f)
    if df is None: return None
    pcol = _pick(df, *_C["P"])
    if pcol is None: return None
    P = pd.to_numeric(df[pcol], errors="coerce").to_numpy(float)
    out = {"P": P, "bottom": float(np.nanmax(P)) if np.isfinite(P).any() else np.nan}
    for v in ("temperature","salinity","sigma_theta","oxygen"):
        c = _pick(df, *_C[v])
        out[v] = pd.to_numeric(df[c], errors="coerce").to_numpy(float) if c else None
    return out

def load_nitrate_profile(cast):
    hits = sorted(SUNA_DIR.glob(f"{cast}*SUNA*1s*.csv")) if SUNA_DIR.exists() else []
    if not hits: return None
    try: d = pd.read_csv(hits[0])
    except Exception: return None
    pc = next((c for c in d.columns if re.search(r"pr(dm|sm)?$|pressure", c, re.I)), None)
    nc = next((c for c in d.columns if re.search(r"no3_qc|nitrate|no3", c, re.I)), None)
    if pc is None or nc is None: return None
    dd = d[[pc, nc]].apply(pd.to_numeric, errors="coerce").dropna()
    if dd.empty: return None
    b = (dd[pc] / GRID_DZ).round() * GRID_DZ
    g = dd.groupby(b)[nc].median()
    return {"P": g.index.to_numpy(float), "nitrate": g.to_numpy(float), "bottom": float(g.index.max())}

In [ ]:
# ------------------------------------------------------------------ positions (cast summary or stations.csv)
def _cast_id_for(n, known):
    for c in known:
        m = re.search(r"(\d+)\s*$", c)
        if m and int(m.group(1)) == int(n): return c
    return f"{CRUISE_ID}_CTD_{int(n):02d}"

def _start_pos(row, which):
    """Start-of-cast fix only. The _end fix is set aside for now (Pierre to resolve); only
    latitude_start / longitude_start are used. A dropped-to-zero latitude is treated as missing."""
    v = pd.to_numeric(row.get(which + "_start"), errors="coerce")
    if pd.isna(v): return np.nan
    if which == "latitude" and abs(v) < 1.0: return np.nan
    return float(v)

def _find_station_file():
    for name in STATION_CANDIDATES:
        p = META_DIR / name
        if p.exists(): return p
    return None

def line_key(station):
    k = re.sub(r"\d+", "", str(station)).strip(" _-")
    return k if k else "LINE"

def load_stations():
    casts = cast_ids()
    if not casts:
        raise FileNotFoundError(f"No *_1m_down.cnv in {ONE_DOWN_DIR} - run step01 Bin Average first.")
    path = _find_station_file()
    if path is None:
        tmpl = META_DIR / "stations_TEMPLATE.csv"; META_DIR.mkdir(parents=True, exist_ok=True)
        pd.DataFrame({"cruise_id": CRUISE_ID, "cast_id": casts, "station": "",
                      "lat_deg": "", "lon_deg": ""}).to_csv(tmpl, index=False)
        raise FileNotFoundError(
            f"No position table found in {META_DIR} (looked for {STATION_CANDIDATES}). Wrote a template "
            f"to {tmpl} - fill lat_deg / lon_deg (decimal degrees, S and W negative) and re-run, or drop "
            f"in the cruise cast-summary csv.")
    s = pd.read_csv(path); s.columns = [c.strip().lower() for c in s.columns]
    if "cast_id" in s.columns:
        s["cast_id"] = s["cast_id"].astype(str)
    elif "cast" in s.columns:
        s["cast_id"] = [_cast_id_for(n, casts) for n in s["cast"]]
    else:
        raise ValueError("position table needs a 'cast' or 'cast_id' column.")
    if {"latitude_start", "longitude_start"}.issubset(s.columns):
        s["lat"] = s.apply(lambda r: _start_pos(r, "latitude"), axis=1)
        s["lon"] = s.apply(lambda r: _start_pos(r, "longitude"), axis=1)
    else:
        latc = next((c for c in s.columns if c in ("lat_deg","lat","latitude")), None)
        lonc = next((c for c in s.columns if c in ("lon_deg","lon","longitude")), None)
        if latc is None or lonc is None:
            raise ValueError("position table needs latitude_start/longitude_start or lat_deg/lon_deg.")
        s["lat"] = pd.to_numeric(s[latc], errors="coerce")
        s["lon"] = pd.to_numeric(s[lonc], errors="coerce")
    # date (day) from the start datetime, for GROUP_BY="date"
    dtc = next((c for c in s.columns if c in ("date_time_utc_start","datetime_start","date_start")), None)
    if dtc is None:
        dtc = next((c for c in s.columns if "start" in c and ("date" in c or "time" in c)), None)
    s["date"] = (pd.to_datetime(s[dtc], errors="coerce").dt.strftime("%Y-%m-%d") if dtc else "")
    if "station" not in s.columns: s["station"] = ""
    s = s[s["cast_id"].isin(casts)].dropna(subset=["lat", "lon"]).reset_index(drop=True)
    if len(s) < 2:
        raise ValueError("need at least 2 casts with valid lat/lon and a matching 1 m profile.")
    s["line"] = s["station"].map(line_key)
    return s[["cast_id", "station", "line", "date", "lat", "lon"]]

In [ ]:
# ------------------------------------------------------------------ geometry + gridding
def _local_km(lat, lon, lat0, lon0):
    return ((np.asarray(lon)-lon0)*np.cos(np.radians(lat0))*111.320,
            (np.asarray(lat)-lat0)*110.574)

def transect_axis(lat, lon):
    lat0, lon0 = float(np.mean(lat)), float(np.mean(lon))
    x, y = _local_km(lat, lon, lat0, lon0)
    Pc = np.column_stack([x, y]); Pc = Pc - Pc.mean(0)
    _, _, Vt = np.linalg.svd(Pc, full_matrices=False)
    u = Vt[0] / np.linalg.norm(Vt[0])
    return lat0, lon0, u

def project(lat, lon, lat0, lon0, u):
    x, y = _local_km(lat, lon, lat0, lon0)
    Pc = np.column_stack([x, y])
    return Pc @ u, Pc @ np.array([-u[1], u[0]])

def build_transect(stations, profiles):
    s = stations.copy()
    s["bottom"] = [profiles[c]["bottom"] if (c in profiles and profiles[c]) else np.nan for c in s["cast_id"]]
    s = s.dropna(subset=["bottom"]).reset_index(drop=True)
    if len(s) < 2: return s.assign(along_km=[], cross_km=[])
    lat0, lon0, u = transect_axis(s["lat"].to_numpy(), s["lon"].to_numpy())
    along, cross = project(s["lat"].to_numpy(), s["lon"].to_numpy(), lat0, lon0, u)
    if np.ptp(s["bottom"]) > 0 and np.corrcoef(along, s["bottom"])[0, 1] < 0:
        along = -along
    s["along_km"] = along - np.nanmin(along)
    s["cross_km"] = cross
    if MANUAL_CAST_ORDER:
        order = [c for c in MANUAL_CAST_ORDER if c in set(s["cast_id"])]
        s = s.set_index("cast_id").loc[order].reset_index()
        xy = [np.array(_local_km(r.lat, r.lon, lat0, lon0)) for r in s.itertuples()]
        s["along_km"] = np.concatenate([[0.0], np.cumsum([np.hypot(*(xy[i]-xy[i-1])) for i in range(1, len(xy))])])
    return s.sort_values("along_km").reset_index(drop=True)

def grid_section(along, profP, profV, seabed, dz=GRID_DZ, nx=GRID_NX):
    xs, ys, vs = [], [], []
    for a, p, v in zip(along, profP, profV):
        p = np.asarray(p, float); v = np.asarray(v, float)
        m = np.isfinite(p) & np.isfinite(v)
        if m.sum(): xs.append(np.full(m.sum(), a)); ys.append(p[m]); vs.append(v[m])
    if len(xs) < 2: return None
    xs = np.concatenate(xs); ys = np.concatenate(ys); vs = np.concatenate(vs)
    if np.unique(xs).size < 2 or np.unique(ys).size < 2: return None
    Xg = np.linspace(float(np.min(along)), float(np.max(along)), nx)
    Pg = np.arange(0.0, float(np.nanmax(ys)) + dz, dz)
    xr = max(xs.max()-xs.min(), 1e-9); yr = max(ys.max()-ys.min(), 1e-9)
    tri = Triangulation((xs-xs.min())/xr, (ys-ys.min())/yr)
    interp = LinearTriInterpolator(tri, vs)
    XX, PP = np.meshgrid(Xg, Pg)
    V = interp((XX-xs.min())/xr, (PP-ys.min())/yr)
    order = np.argsort(along)
    sb = np.interp(Xg, np.asarray(along)[order], np.asarray(seabed)[order],
                   left=np.asarray(seabed)[order][0], right=np.asarray(seabed)[order][-1])
    V = np.ma.masked_where(PP > sb[None, :] + dz, V)
    return Xg, Pg, V, sb

In [ ]:
# ------------------------------------------------------------------ plot one section
def plot_section(var, transect, profiles, label=""):
    st = VAR_STYLE.get(var, dict(cmap="viridis", unit="", label=var))
    profP, profV, seabed, have = [], [], [], []
    for _, r in transect.iterrows():
        cast = r["cast_id"]
        if var == "nitrate":
            pr = load_nitrate_profile(cast)
            if pr is None: continue
            profP.append(pr["P"]); profV.append(pr["nitrate"])
        else:
            pr = profiles.get(cast)
            if pr is None or pr.get(var) is None: continue
            profP.append(pr["P"]); profV.append(pr[var])
        seabed.append(r["bottom"]); have.append(r["along_km"])
    if len(profP) < 2:
        print(f"    {var}: fewer than 2 casts have this variable - skipped."); return None
    g = grid_section(np.array(have), profP, profV, seabed)
    if g is None:
        print(f"    {var}: could not grid - skipped."); return None
    Xg, Pg, V, sb = g
    fig, ax = plt.subplots(figsize=(9.2, 5.0))
    cf = ax.contourf(Xg, Pg, V, levels=14, cmap=st["cmap"])
    cl = ax.contour(Xg, Pg, V, levels=14, colors="k", linewidths=0.3, alpha=0.5)
    ax.clabel(cl, inline=True, fontsize=6, fmt="%g")
    ax.plot(Xg, sb, color="k", lw=1.0)
    ax.fill_between(Xg, sb, Pg.max(), color="#d9cbb3", zorder=3)
    ax.scatter(have, np.zeros_like(have), marker="v", color="k", s=28, zorder=5, clip_on=False)
    for _, r in transect.iterrows():
        lab = str(r.get("station") or r["cast_id"].split("_")[-1])
        ax.annotate(lab, (r["along_km"], 0), xytext=(0, 6), textcoords="offset points",
                    ha="center", fontsize=6.5, rotation=90, va="bottom", annotation_clip=False)
    ax.set_ylim(Pg.max(), 0)
    ax.set_xlabel("distance offshore (km)"); ax.set_ylabel("pressure (dbar)")
    ttl = f"{CRUISE_ID}  -  {st['label']} cross-shelf section" + (f"  ({label})" if label else "")
    ax.set_title(ttl, fontsize=11, pad=26)
    cb = fig.colorbar(cf, ax=ax, pad=0.02); cb.set_label(st["unit"] or st["label"])
    fig.text(0.01, 0.01, "Colour between stations is interpolation (a visual aid), not measured data. "
             "Station ticks mark the real casts.", fontsize=7, color="#666666")
    fig.tight_layout(rect=[0, 0.03, 1, 0.96])
    tag = (re.sub(r"[^0-9A-Za-z_-]", "", label) + "__") if label else ""
    png = OUT_DIR / f"{CRUISE_ID}__section__{tag}{var}.png"
    fig.savefig(png, dpi=150, bbox_inches="tight"); plt.show()
    pd.DataFrame(np.ma.filled(V, np.nan), index=np.round(Pg, 2),
                 columns=np.round(Xg, 3)).to_csv(OUT_DIR / f"{CRUISE_ID}__section__{tag}{var}.csv",
                                                 encoding="utf-8-sig")
    print(f"    {var}: {len(profP)} casts -> {png.name}")
    return png

In [ ]:
# ------------------------------------------------------------------ run
stations = load_stations()
profiles = {c: load_profile(c) for c in stations["cast_id"]}

# build the groups (one section each)
if SECTION_ALL or MANUAL_CAST_ORDER:
    groups = {"": stations}
elif GROUP_BY == "date":
    keys = DATES_TO_SECTION or sorted(d for d in stations["date"].dropna().unique() if d)
    groups = {d: stations[stations["date"] == d] for d in keys}
    print("casts per day:"); print(stations.groupby("date")["cast_id"].count().to_string())
elif GROUP_BY == "line":
    counts = stations.groupby("line")["cast_id"].count().sort_values(ascending=False)
    print("casts per line:"); print(counts.to_string())
    keys = LINES_TO_SECTION or [ln for ln in counts.index if counts[ln] >= MIN_GROUP_CASTS]
    groups = {ln: stations[stations["line"] == ln] for ln in keys}
else:
    groups = {"": stations}

for key, sub in groups.items():
    if not (SECTION_ALL or MANUAL_CAST_ORDER) and len(sub) < MIN_GROUP_CASTS:
        print(f"\n[{key or 'all'}] {len(sub)} cast(s) < MIN_GROUP_CASTS={MIN_GROUP_CASTS} - skipped."); continue
    transect = build_transect(sub, profiles)
    if len(transect) < 2:
        print(f"\n[{key or 'all'}] fewer than 2 casts with a profile - skipped."); continue
    span = float(transect["along_km"].max() - transect["along_km"].min())
    if span < MIN_SPAN_KM and not (SECTION_ALL or MANUAL_CAST_ORDER):
        print(f"\n[{key or 'all'}] along-track span {span:.1f} km < {MIN_SPAN_KM} km - looks like a "
              f"single spot / repeat station, not a transect; skipped."); continue
    print(f"\n[{key or 'all'}] {len(transect)} casts, span {span:.1f} km, offshore order:")
    print(transect[["cast_id","station","lat","lon","along_km","cross_km","bottom"]]
          .round({"lat":4,"lon":4,"along_km":1,"cross_km":2,"bottom":1}).to_string(index=False))
    off = transect[transect["cross_km"].abs() > CROSS_WARN_KM]
    if len(off):
        print(f"  NOTE: {len(off)} station(s) > {CROSS_WARN_KM:g} km off the fitted line "
              f"({', '.join(off['cast_id'])}); read those columns with care.")
    for v in VARS:
        try: plot_section(v, transect, profiles, label=key)
        except Exception as e: print(f"    {v}: error - {e}")
print("\ndone ->", OUT_DIR)

## 7. Interactive section explorer

In [ ]:
# ------------------------------------------------------------------ interactive section explorer
try:
    import ipywidgets as W
    from IPython.display import display as _disp, clear_output
    try: get_ipython().run_line_magic("matplotlib","inline")
    except Exception: pass
    _HAVE_W=True
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets then Restart + Run All.")

if _HAVE_W:
    if GROUP_BY=="line": _keys=sorted(stations["line"].dropna().unique().tolist())
    elif GROUP_BY=="date": _keys=sorted(d for d in stations["date"].dropna().unique() if d)
    else: _keys=["all"]
    if not _keys: _keys=["all"]
    wg=W.Dropdown(options=_keys,value=_keys[0],description="section")
    wv=W.Dropdown(options=VARS,value=VARS[0],description="variable")
    eout=W.Output()
    def _sub(key):
        if key=="all": return stations
        return stations[stations["date"]==key] if GROUP_BY=="date" else stations[stations["line"]==key]
    def _r(*_):
        with eout:
            clear_output(wait=True)
            sub=_sub(wg.value)
            if len(sub)<2: print("fewer than 2 casts in this section"); return
            tr=build_transect(sub,profiles)
            if len(tr)<2: print("fewer than 2 casts with a profile"); return
            plot_section(wv.value,tr,profiles,label=("" if wg.value=="all" else str(wg.value)))
    wg.observe(_r,"value"); wv.observe(_r,"value")
    _disp(W.VBox([W.HBox([wg,wv]),eout])); _r()